## Part 1.1: Load and Verify Project Configuration

### Objective

Load the project configuration from `credentials.json` and `config.json` and confirm that all required variables are available before running any database or data-processing steps.

Because this notebook runs in Google Colab, it cannot directly access files stored in the local project repository. The two JSON files must therefore be uploaded from the local machine into the temporary Colab runtime.

### Configuration Files

* `credentials.json` contains sensitive connection information, such as database credentials.
* `config.json` contains reusable project settings, including Google Drive paths and audit-log locations.

### Process

This step will:

1. Prompt the user to upload both JSON files from the local machine.
2. Confirm that the required files were uploaded.
3. Load the JSON contents into Python dictionaries.
4. Verify that the configuration variables are accessible.
5. Mask sensitive values when displaying credentials.
6. Remove the uploaded JSON files from the Colab runtime after loading them into memory.

Removing the temporary files reduces unnecessary exposure of credentials within the Colab runtime. The loaded `credentials` and `config` dictionaries will remain available in memory for subsequent notebook steps.

### Expected Outcome

* Both JSON files are uploaded and loaded successfully.
* Configuration values are accessible through the `credentials` and `config` dictionaries.
* Sensitive values are not displayed in plain text.
* Temporary copies of the JSON files are removed from the Colab runtime.
* The notebook is ready to use the configuration in later pipeline stages.


In [ ]:
# ==========================================
# Part 1.1: Upload, Load, and Verify
# Project Configuration in Google Colab
# ==========================================

import json
from pathlib import Path
from google.colab import files


REQUIRED_FILES = {
    "credentials.json",
    "config.json",
}


def upload_configuration_files() -> None:
    """
    Prompt the user to upload credentials.json and config.json
    from the local machine into the temporary Colab runtime.
    """
    print("Upload the following files:")
    print("- credentials.json")
    print("- config.json")
    print()

    uploaded_files = files.upload()
    uploaded_names = set(uploaded_files.keys())

    missing_files = REQUIRED_FILES - uploaded_names

    if missing_files:
        raise FileNotFoundError(
            "Missing required file(s): "
            + ", ".join(sorted(missing_files))
        )

    print("\nConfiguration files uploaded successfully.")


def load_json(file_path: Path) -> dict:
    """
    Load a JSON file and return its contents as a dictionary.
    """
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    try:
        with file_path.open("r", encoding="utf-8") as file:
            data = json.load(file)

    except json.JSONDecodeError as error:
        raise ValueError(
            f"Invalid JSON format in {file_path.name}: {error}"
        ) from error

    if not isinstance(data, dict):
        raise TypeError(
            f"{file_path.name} must contain a JSON object."
        )

    return data


def mask_sensitive_value(key: str, value):
    """
    Mask sensitive values before displaying them.
    """
    sensitive_terms = (
        "PASS",
        "PASSWORD",
        "SECRET",
        "TOKEN",
        "API_KEY",
        "PRIVATE_KEY",
    )

    key_upper = key.upper()

    if any(term in key_upper for term in sensitive_terms):
        return "********"

    return value


def display_configuration(
    file_name: str,
    data: dict,
    mask_sensitive: bool = False,
) -> None:
    """
    Print configuration variables for verification.
    """
    print(f"========== {file_name} ==========")

    for key, value in data.items():
        displayed_value = (
            mask_sensitive_value(key, value)
            if mask_sensitive
            else value
        )

        print(f"{key:<25}: {displayed_value}")

    print()


def remove_runtime_file(file_path: Path) -> None:
    """
    Delete an uploaded file from the temporary Colab filesystem.
    """
    if file_path.exists():
        file_path.unlink()
        print(f"Removed temporary file: {file_path.name}")


# ------------------------------------------
# Upload files from local machine
# ------------------------------------------

upload_configuration_files()


# ------------------------------------------
# Define temporary Colab file paths
# ------------------------------------------

CREDENTIALS_FILE = Path("/content/credentials.json")
CONFIG_FILE = Path("/content/config.json")


# ------------------------------------------
# Load JSON configuration into memory
# ------------------------------------------

credentials = load_json(CREDENTIALS_FILE)
config = load_json(CONFIG_FILE)

print("\ncredentials.json loaded successfully.")
print("config.json loaded successfully.\n")


# ------------------------------------------
# Verify loaded variables
# ------------------------------------------

display_configuration(
    file_name="credentials.json",
    data=credentials,
    mask_sensitive=True,
)

display_configuration(
    file_name="config.json",
    data=config,
    mask_sensitive=False,
)


# ------------------------------------------
# Remove uploaded files from Colab storage
# ------------------------------------------

remove_runtime_file(CREDENTIALS_FILE)
remove_runtime_file(CONFIG_FILE)


# ------------------------------------------
# Final verification
# ------------------------------------------

print()
print("=" * 60)
print("Project configuration loaded into memory successfully.")
print("Temporary JSON files were removed from the Colab runtime.")
print("Configuration variables remain accessible in this session.")
print("=" * 60)

## Part 1.2: Mount Google Drive and Verify Project Paths

### Objective

Mount Google Drive in the Colab runtime and verify that the project directories and audit-log file defined in `config.json` exist at the expected locations.

The configuration variables loaded in Part 1.1 are used to construct the required Google Drive paths.

### Expected Project Structure

```text
BASE_DIR/
└── logs/
    └── pipeline_audit.log
```

The configured values represent different types of filesystem objects:

* `BASE_DIR` identifies the main project directory in Google Drive.
* `LOG_AUDIT` identifies the audit-log directory inside `BASE_DIR`.
* `PIPELINE_AUDIT` identifies the pipeline audit-log filename inside `LOG_AUDIT_PATH`.

After resolving the configuration values, the notebook creates the following Python path variables:

```text
BASE_DIR_PATH
LOG_AUDIT_PATH
PIPELINE_AUDIT_PATH
```

### Process

This step will:

1. Mount Google Drive at `/content/drive`.
2. Read `BASE_DIR`, `LOG_AUDIT`, and `PIPELINE_AUDIT` from the previously loaded `config` dictionary.
3. Convert the configured values into complete Colab filesystem paths.
4. Verify that `BASE_DIR_PATH` exists and is a directory.
5. Verify that `LOG_AUDIT_PATH` exists and is a directory.
6. Check whether `PIPELINE_AUDIT_PATH` exists and is a file.

The pipeline audit file may not exist before logging is initialized. Therefore, a missing `pipeline_audit.log` file is reported as a warning rather than treated as an immediate configuration failure. However, if the configured path exists as a directory instead of a file, the verification will fail.

### Expected Outcome

* Google Drive is mounted successfully.
* `BASE_DIR_PATH` points to an existing project directory.
* `LOG_AUDIT_PATH` points to the existing `logs` directory.
* `PIPELINE_AUDIT_PATH` points to `pipeline_audit.log`.
* The notebook is ready to use the verified paths for logging and subsequent pipeline operations.


In [ ]:
# ==========================================
# Part 1.2: Mount Google Drive and Verify
# Project Directory Structure
# ==========================================

from pathlib import Path
from google.colab import drive


# ------------------------------------------
# Mount Google Drive
# ------------------------------------------

DRIVE_MOUNT_POINT = Path("/content/drive")
drive.mount(str(DRIVE_MOUNT_POINT))

print("\n✓ Google Drive mounted successfully.\n")


# ------------------------------------------
# Read Configuration
# ------------------------------------------

BASE_DIR = config["BASE_DIR"]
LOG_AUDIT_PATH = config["LOG_AUDIT_PATH"]
PIPELINE_AUDIT = config["PIPELINE_AUDIT"]


# ------------------------------------------
# Resolve Google Drive Paths
# ------------------------------------------

MY_DRIVE = DRIVE_MOUNT_POINT / "MyDrive"


def resolve_drive_path(path_value: str) -> Path:
    """
    Convert a config path into a Google Drive path.
    """

    path_value = path_value.strip().replace("\\", "/")

    if path_value.startswith("/MyDrive/"):
        return MY_DRIVE / path_value.replace("/MyDrive/", "", 1)

    if path_value.startswith("MyDrive/"):
        return MY_DRIVE / path_value.replace("MyDrive/", "", 1)

    return MY_DRIVE / path_value


BASE_DIR_PATH = resolve_drive_path(BASE_DIR)
LOG_AUDIT_PATH = BASE_DIR_PATH / LOG_AUDIT_PATH
PIPELINE_AUDIT_PATH = LOG_AUDIT_PATH / PIPELINE_AUDIT


# ------------------------------------------
# Verify Project Structure
# ------------------------------------------

print("========== Google Drive Verification ==========\n")

# BASE_DIR
assert BASE_DIR_PATH.exists(), f"Missing directory: {BASE_DIR_PATH}"
assert BASE_DIR_PATH.is_dir(), f"Not a directory: {BASE_DIR_PATH}"

print(f"✓ BASE_DIR_PATH")
print(f"  {BASE_DIR_PATH}\n")


# LOG_AUDIT_PATH
assert LOG_AUDIT_PATH.exists(), f"Missing directory: {LOG_AUDIT_PATH}"
assert LOG_AUDIT_PATH.is_dir(), f"Not a directory: {LOG_AUDIT_PATH}"

print(f"✓ LOG_AUDIT_PATH")
print(f"  {LOG_AUDIT_PATH}\n")


# PIPELINE_AUDIT_PATH (file)
if PIPELINE_AUDIT_PATH.exists():

    assert PIPELINE_AUDIT_PATH.is_file(), (
        f"Expected a file but found a directory:\n{PIPELINE_AUDIT_PATH}"
    )

    print(f"✓ PIPELINE_AUDIT_PATH")
    print(f"  {PIPELINE_AUDIT_PATH}\n")

else:

    print("⚠ PIPELINE_AUDIT_PATH")
    print(f"  {PIPELINE_AUDIT_PATH}")
    print("  Log file does not exist yet (this is acceptable).\n")


print("=" * 55)
print("Google Drive verification completed successfully.")
print("=" * 55)